# 宏观场景推演模型 — 交互式演示

**人民币升值 → 通胀/出口/行业利润 → 资产价格重估**

本 Notebook 提供交互式参数调节，实时观察传导链条的变化。

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(__file__))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Inline')
import matplotlib.pyplot as plt

from model import MacroParams, ScenarioEngine

%matplotlib inline
plt.rcParams['font.family'] = 'WenQuanYi Micro Hei'
plt.rcParams['figure.dpi'] = 120

## 一、调节核心参数

修改下方单元格中的 `apprec_rate` 即可切换升值速度。

In [ ]:
# ═══ 可调参数 ═══
apprec_rate = 0.06      # 年化升值幅度 (6%)
n_years = 5
n_sims = 2000
# ═════════════════

params = MacroParams(
    cny_annual_apprec=apprec_rate,
    n_years=n_years,
    n_simulations=n_sims,
    seed=42,
)

engine = ScenarioEngine(params)
results = engine.run()
print('✓ 模拟完成')

## 二、汇率路径 (分位数带)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
path = results['fx_path']
years = range(path.shape[1])
ax.plot(years, np.median(path, axis=0), 'b-', lw=2, label='中位数')
ax.fill_between(years, np.percentile(path, 25, axis=0), np.percentile(path, 75, axis=0), alpha=0.3, color='blue')
ax.axhline(7.20, color='gray', ls='--', alpha=0.7, label='基准7.20')
ax.set_title(f'USD/CNY 路径 (年化升值 {apprec_rate:.0%})')
ax.set_xlabel('年'); ax.set_ylabel('USD/CNY')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

final_apprec = np.median(results['appreciation'], axis=0)[-1]
print(f'5年累计升值: {final_apprec:.1f}%')
print(f'USD/CNY 终值: {np.median(path, axis=0)[-1]:.2f}')

## 三、行业利润冲击热力图

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
data = results['industry_profit'].T.values
im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=-0.15, vmax=0.15)
ax.set_xticks(range(len(results['industry_profit'])))
ax.set_xticklabels(results['industry_profit'].index)
ax.set_yticks(range(len(data)))
ax.set_yticklabels(results['industry_profit'].columns, fontsize=9)
for i in range(len(data)):
    for j in range(len(data[i])):
        ax.text(j, i, f"{data[i][j]:.1%}", ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8).set_label('利润冲击')
ax.set_title('行业利润冲击 (行=行业, 列=年份)')
plt.tight_layout()
plt.show()

## 四、资产年化收益对比

In [ ]:
asset_data = {
    'A股(加权)': float(np.median(results['equity_return'])),
    '中国国债': float(results['bond_return']),
    '黄金': float(np.median(results['gold_return'])),
    '工业金属': float(np.median(results['commodity_return'])),
}

fig, ax = plt.subplots(figsize=(8, 5))
names = list(asset_data.keys())
vals = list(asset_data.values())
colors = ['#3498db', '#95a5a6', '#f39c12', '#e67e22']
bars = ax.bar(names, vals, color=colors, alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')
ax.set_title('资产年化收益 (5年中位)')
ax.grid(alpha=0.3, axis='y')
plt.show()

print('资产年化收益:')
for k, v in asset_data.items():
    print(f'  {k:10s} {v:+.1%}')

## 五、敏感性分析 (交互式)

修改上方 `apprec_rate` 为不同值 (0.0 / 0.03 / 0.06 / 0.10)，重新运行即可对比。